# Session 1: NumPy Fundamentals (2 hours)

**Goal:** Build the mental model: *read an ML-style task → map it to array shapes and operations → implement in NumPy.* We start with real mini-tasks (cases), not a long list of basics.

| Section | Time | Focus |
|---------|------|-------|
| Setup | 10 min | Import & verify environment |
| Anchor case 1 | 40 min | Weighted sum & batch prediction |
| Anchor case 2 | 30 min | Normalise a vector to 0–1 |
| Anchor case 3 | 35 min | Boolean indexing (extract with mask) |
| Anchor case 4 | 35 min | View vs copy — avoid the bug |
| Reference | 50 min | NumPy essentials drill (if you need more practice) |

## Setup (10 min)

In [1]:
import numpy as np

print(f"NumPy version: {np.__version__}")
print("Setup complete!")

NumPy version: 1.26.4
Setup complete!


---
## Anchor case 1: Weighted sum and batch prediction [EASY]

In ML we often compute **weighted sums**: one number per example (e.g. a score or prediction). Here we do it for **one** example, then for a **batch** of examples (matrix × vector).

### Concept: Arrays and shape

An **array** is a grid of numbers. **Shape** is the size along each dimension: e.g. `(5,)` = 1D vector of length 5, `(3, 4)` = 3 rows, 4 columns. Use `arr.shape` to inspect. Create from a list with `np.array([...])`.

### Worked example: One neuron, then a batch

**Task:** Given scores (features) and weights, compute the weighted sum (like one neuron). Then do it for 5 examples at once: `batch` has shape (5, 2), `weights` (2,). We want one number per example → shape (5,).

**Expert reasoning:** For one example, `scores @ weights` or `np.dot(scores, weights)` gives a scalar. For a batch, we need matrix × vector: (5, 2) @ (2,) → (5,). NumPy's `@` does exactly that. We keep `weights` as (2,) so the last dimension of `batch` (2) matches; the result loses that dimension → (5,).

In [ ]:
# One example: 2 features, 2 weights
scores = np.array([0.5, 1.5])   # shape (2,)
weights = np.array([2.0, -0.5]) # shape (2,)
one_pred = np.dot(scores, weights)  # or scores @ weights → scalar
print(f"One prediction: {one_pred}, shape {np.array(one_pred).shape}")

# Batch of 5 examples, 2 features each
batch = np.array([[0.5, 1.5], [1.0, 0.0], [0.0, 1.0], [2.0, 2.0], [0.1, 0.9]])  # (5, 2)
batch_pred = batch @ weights  # (5, 2) @ (2,) → (5,)
print(f"Batch predictions: {batch_pred}")
print(f"Shapes: batch {batch.shape}, weights {weights.shape}, result {batch_pred.shape}")

### Your turn [EASY]

Use the same idea: 10 examples, **3** features. `batch_10` has shape (10, 3), `w` has shape (3,). Compute the vector of 10 predictions (shape (10,)).

In [ ]:
np.random.seed(42)
batch_10 = np.random.randn(10, 3)  # 10 examples, 3 features
w = np.array([1.0, -0.5, 0.0])

# YOUR CODE HERE: compute predictions, shape (10,)
predictions_10 = ...

print(f"predictions_10.shape = {predictions_10.shape}")
assert predictions_10.shape == (10,), f"Expected (10,), got {predictions_10.shape}"

<details>
<summary>💡 Click to see solution</summary>

```python
predictions_10 = batch_10 @ w
```
</details>


### Sensemaking

**What would break if `weights` had shape (3,) and `batch` had shape (5, 2)?** Write one sentence (e.g. dimension mismatch, which dimensions?).

_Your answer: the inner dimensions don't match: (5,2) @ (3,) — 2 ≠ 3, so NumPy would raise an error. In general we need the last dim of the left array to equal the only dim of the right vector._

---
## Anchor case 2: Normalise a vector to 0–1 [EASY]

**Task:** Given a vector of numbers, rescale so the minimum becomes 0 and the maximum becomes 1. Formula: `(x - min) / (max - min)`.

### Worked example

In [ ]:
v = np.array([10.0, 20.0, 5.0, 35.0, 15.0])
v_min = v.min()
v_max = v.max()
v_normalized = (v - v_min) / (v_max - v_min)
print(f"Original: {v}")
print(f"Normalized: {v_normalized}")
print(f"Min/max of normalized: {v_normalized.min():.2f}, {v_normalized.max():.2f}")

### Your turn [EASY]

Normalise a **random** vector of length 7 (use `np.random.rand(7)`). Check that the result has min 0 and max 1.

In [ ]:
np.random.seed(0)
vec = np.random.rand(7)
# YOUR CODE HERE
vec_norm = ...
print(vec_norm)
assert np.isclose(vec_norm.min(), 0) and np.isclose(vec_norm.max(), 1), "Min should be 0, max should be 1"

<details>
<summary>💡 Click to see solution</summary>

```python
vec_norm = (vec - vec.min()) / (vec.max() - vec.min())
```
</details>


### Sensemaking

**When would you use `reshape(-1, 1)` on a 1D vector in an ML context?** (One sentence: what shape are we aiming for and why?)

_Your answer: e.g. to get a column vector (n, 1) so that when we multiply or add with a matrix of shape (n, d), broadcasting works correctly (e.g. adding a bias per feature)._

---
## Anchor case 3: Extract elements with a mask (boolean indexing) [MEDIUM]

**Task:** We have predictions (n,) and a boolean mask `valid` (n,) indicating which examples are valid (e.g. no missing label). Extract only the predictions where `valid` is True. Then compute the mean of **only those** predictions.

**Why it matters:** In ML we often filter out invalid or padded positions before computing a loss or metric.

### Worked example

In [ ]:
predictions = np.array([0.1, 0.9, 0.3, 0.7, 0.5])
valid = np.array([True, True, False, True, False])  # 3 valid
pred_valid = predictions[valid]   # boolean indexing → 1D array of length 3
mean_valid = pred_valid.mean()
print(f"predictions: {predictions}")
print(f"valid:       {valid}")
print(f"pred_valid: {pred_valid}")
print(f"mean_valid: {mean_valid}")

### Your turn [MEDIUM]

Given `scores` of shape (20,) and `mask` of shape (20,) (boolean), compute the **maximum** score among the positions where `mask` is True.

In [ ]:
np.random.seed(42)
scores = np.random.rand(20)
mask = np.random.rand(20) > 0.5  # about half True
# YOUR CODE HERE
max_valid = ...
print(f"Number of valid: {mask.sum()}, max_valid: {max_valid}")

<details>
<summary>💡 Click to see solution</summary>

```python
max_valid = scores[mask].max()
```
</details>


### Sensemaking

**What is the shape of `predictions[valid]` if `predictions` has shape (100,) and `valid` has 30 True values?**

_Your answer: (30,) — we get one element per True in the mask._

---
## Anchor case 4: View vs copy — avoid a bug [MEDIUM]

**Task:** You take a **slice** of an array (e.g. first 3 elements), modify that slice in-place, and expect the original array to be unchanged. Run the code below: does the original change? Then fix it so the original stays unchanged (use `.copy()` when you need an independent array).

### Worked example: the bug

In [ ]:
original = np.array([10, 20, 30, 40, 50])
slice_view = original[1:4]   # slice → view (shares memory)
slice_view[0] = 999
print(f"After modifying slice_view: original = {original}")
print("The original changed! Slicing returns a VIEW.")

### Your turn [MEDIUM]

Create a **copy** of the first 3 elements, modify the copy (e.g. set copy[0] = -1), and verify that `original` is unchanged.

In [ ]:
original = np.array([10, 20, 30, 40, 50])
# YOUR CODE: get first 3 elements as a COPY, then set copy[0] = -1
first_three = ...  # hint: use .copy()
first_three[0] = -1
print(f"first_three: {first_three}")
print(f"original:    {original}")
assert original[0] == 10, "Original should be unchanged"

<details>
<summary>💡 Click to see solution</summary>

```python
first_three = original[:3].copy()
```
</details>


### Sensemaking

**When would you prefer a view over a copy?** (Think: memory, speed, safety.)

_Your answer: View when you only read and want to save memory/speed. Copy when you will modify and must not change the original._

---
## Reference: NumPy essentials (if you need more drill)

The anchor cases above cover the mental model we need for ML. Below are exercises from [NumPy-100](https://github.com/rougier/numpy-100) (numpy.org). **Goal: Complete 80+ exercises** for strong foundations. Full solutions: [100_Numpy_exercises_with_solutions](https://github.com/rougier/numpy-100/blob/master/100_Numpy_exercises_with_solutions.md). For each one, think: *"How would I do this with a for loop? Why is the NumPy version faster?"*

### Group A: Basic Array Operations (Exercises 1-10)

**Q1.** Create a null vector of size 10

In [26]:
# YOUR CODE HERE
arr = np.zeros(10)
print(arr)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


<details>
<summary>💡 Click to see solution</summary>

```python
z = np.zeros(10)
```
</details>


**Q2.** Create a null vector of size 10 but the fifth value is 1

In [27]:
# YOUR CODE HERE
arr = np.zeros(10)
arr[4] = 1
arr

array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.zeros(10)
z[4] = 1
```
</details>


**Q3.** Create a vector with values ranging from 10 to 49

In [29]:
# YOUR CODE HERE
z = np.arange(10,50)
z

array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.arange(10, 50)
```
</details>


**Q4.** Reverse a vector (first element becomes last)

In [32]:
# YOUR CODE HERE
z = np.arange(10,50)[::-1]
z

array([49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33,
       32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16,
       15, 14, 13, 12, 11, 10])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.arange(10, 50)[::-1]
```
</details>


**Q5.** Create a 3x3 matrix with values ranging from 0 to 8

In [35]:
# YOUR CODE HERE
z = np.arange(9).reshape(3,3)
z

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.arange(9).reshape(3, 3)
```
</details>


**Q6.** Find indices of non-zero elements from `[1, 2, 0, 0, 4, 0]`

In [37]:
# YOUR CODE HERE
z = np.array([1, 2, 0, 0, 4, 0])
idx = np.nonzero(z)[0]
idx

array([0, 1, 4])

<details>
<summary>💡 Click to see solution</summary>

```python
arr = np.array([1, 2, 0, 0, 4, 0])
z = np.nonzero(arr)[0]
```
</details>


**Q7.** Create a 3x3 identity matrix

In [39]:
# YOUR CODE HERE
z = np.eye(3)
z

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.eye(3)
```
</details>


### Group B: Indexing and Slicing (Exercises 15-25)

**Q8.** Create a 2D array with 1 on the border and 0 inside (5x5)

In [41]:
# YOUR CODE HERE
z = np.ones(5*5).reshape(5,5)
z[1:-1,1:-1] = 0
z

array([[1., 1., 1., 1., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 1., 1., 1., 1.]])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.ones((5, 5), dtype=int)
z[1:-1, 1:-1] = 0
```
</details>


**Q9.** Pad an existing array (3x3 of ones) with a border of zeros

In [72]:
# YOUR CODE HERE
z = np.random.randint(0, 9, size=(3, 3))
pad_z = np.pad(z,pad_width=1,mode="constant",constant_values=0)
print(pad_z)

[[0 0 0 0 0]
 [0 0 3 8 0]
 [0 7 7 1 0]
 [0 8 4 7 0]
 [0 0 0 0 0]]


<details>
<summary>💡 Click to see solution</summary>

```python
core = np.ones((3, 3), dtype=int)
z = np.pad(core, pad_width=1, mode="constant", constant_values=0)
```
</details>


**Q10.** Create a 5x5 matrix with row values ranging from 0 to 4

In [81]:
# YOUR CODE HERE
z = np.random.randint(0,5, size=(5,5))
x = np.tile(np.arange(5), (5,1))
x

array([[0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4]])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.tile(np.arange(5), (5, 1))
```
</details>


**Q11.** Create a checkerboard 8x8 matrix using slicing

In [ ]:
# YOUR CODE HERE
z = np.zeros(64, dtype=int).reshape(8,8)
z[0::2,0::2] = 1
z[1::2,1::2] = 1
z


array([[1, 0, 1, 0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0, 1, 0, 1],
       [1, 0, 1, 0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0, 1, 0, 1],
       [1, 0, 1, 0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0, 1, 0, 1],
       [1, 0, 1, 0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0, 1, 0, 1]])

<details>
<summary>💡 Click to see solution</summary>

```python
z = np.zeros((8, 8), dtype=int)
z[0::2, 0::2] = 1
z[1::2, 1::2] = 1
```
</details>


**Q12.** Normalize a 5x5 random matrix (values between 0 and 1)

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
np.random.seed(0)  # or any seed
z = np.random.rand(5, 5)
z = (z - z.min()) / (z.max() - z.min())
```
</details>


### Group C: Array Manipulation (Exercises 30-40)

**Q13.** How to find common values between two arrays?

In [ ]:
A = np.array([1, 2, 3, 4, 5])
B = np.array([3, 4, 5, 6, 7])

# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
common = np.intersect1d(A, B)
```
</details>


**Q14.** Create a random vector of size 10 and sort it

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
z = np.random.rand(10)
z = np.sort(z)
```
</details>


**Q15.** Replace the maximum value by 0 in a random vector

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
z = np.random.rand(10)
z[z.argmax()] = 0
```
</details>


**Q16.** Subtract the mean of each row of a matrix (5x5 random)

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
m = np.random.rand(5, 5)
m = m - m.mean(axis=1, keepdims=True)
```
</details>


**Q17.** Sort an array by the nth column

In [ ]:
# Create a random 5x3 matrix and sort by column index 1
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
m = np.random.rand(5, 3)
m = m[m[:, 1].argsort()]
```
</details>


### Group D: Mathematical Operations (Exercises 45-55)

**Q18.** Compute `((A + B) * (-A / 2))` element-wise without creating intermediate arrays (use `np.add`, `np.multiply`, etc. with `out` parameter)

In [ ]:
A = np.ones(3) * 1
B = np.ones(3) * 2

# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
result = np.empty_like(A, dtype=float)
np.add(A, B, out=result)
np.multiply(result, -A / 2, out=result)
```
</details>


**Q19.** Get the n largest values of an array

In [ ]:
Z = np.arange(10000)
np.random.shuffle(Z)
n = 5

# YOUR CODE HERE (hint: np.argpartition)


<details>
<summary>💡 Click to see solution</summary>

```python
indices = np.argpartition(Z, -n)[-n:]
# Or to get values: Z[np.argpartition(Z, -n)[-n:]]
```
</details>


**Q20.** Compute the sum of a vector using different methods and compare speed

In [ ]:
Z = np.random.random(1000000)

# Method 1: Python sum()
%timeit sum(Z)

# Method 2: np.sum()
%timeit np.sum(Z)

# Method 3: Z.sum()
%timeit Z.sum()

# Method 4: np.add.reduce()
%timeit np.add.reduce(Z)

**Q21.** Compute the dot product of two vectors manually (without np.dot), then verify

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Manual dot product
dot_manual = ...

# Verify with NumPy
dot_numpy = np.dot(a, b)

print(f"Manual: {dot_manual}, NumPy: {dot_numpy}, Match: {dot_manual == dot_numpy}")

<details>
<summary>💡 Click to see solution</summary>

```python
dot_manual = np.sum(a * b)
# Or: dot_manual = (a * b).sum()
```
</details>


**Q22.** Compute running (cumulative) sum of a vector

In [ ]:
Z = np.array([1, 2, 3, 4, 5])

# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
z = np.cumsum(Z)
```
</details>


### Group E: Linear Algebra Basics (Exercises 60-65)

**Q23.** Multiply a 5x3 matrix by a 3x2 matrix (real matrix product)

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
A = np.random.rand(5, 3)
B = np.random.rand(3, 2)
M = A @ B
```
</details>


**Q24.** Create a structured array with `name` (string) and `age` (int) fields

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Click to see solution</summary>

```python
dtype = np.dtype([("name", "U10"), ("age", "i4")])
data = np.array([("Alice", 30), ("Bob", 25)], dtype=dtype)
```
</details>


**Q25.** Given a 1D array, negate all elements between 3 and 8 (in-place)

In [ ]:
Z = np.arange(11)

# YOUR CODE HERE

print(Z)

<details>
<summary>💡 Click to see solution</summary>

```python
Z[(3 < Z) & (Z < 8)] *= -1
```
</details>


---
## Extended NumPy 100 (Q26–Q85) — Goal: 80+ exercises

You have completed Q1–Q25. To reach **80+ exercises**, work through Q26–Q85 below. Each has a solution dropdown. Full list: [NumPy 100](https://numpy.org/doc/stable/user/absolute_beginners.html) | [Solutions](https://github.com/rougier/numpy-100).

### Group F: NumPy quirks & types (Q26–Q30)

**Q26.** What does `sum(range(5), -1)` return? What about after `from numpy import *`?

In [ ]:
# Try: print(sum(range(5), -1)); from numpy import *; print(sum(range(5), -1))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

Python's `sum(range(5), -1)` → 9 (start=-1). NumPy's `sum` treats second arg as axis → different result. Avoid `from numpy import *`.
</details>


**Q27.** Round a float array away from zero (e.g. 2.3→3, -2.3→-3)

In [ ]:
Z = np.random.uniform(-10, 10, 10)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.copysign(np.ceil(np.abs(Z)), Z)
```
</details>


**Q28.** Create a 5x5 matrix with 1,2,3,4 just below the diagonal

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.diag(1 + np.arange(4), k=-1)
```
</details>


**Q29.** For a (6,7,8) array, what is the (x,y,z) index of the 100th element?

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.unravel_index(99, (6, 7, 8))
```
</details>


**Q30.** Create a vector of size 10 with values in (0, 1), both excluded

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.linspace(0, 1, 11, endpoint=False)[1:]
```
</details>


### Group G: Arrays & comparisons (Q31–Q40)

**Q31.** Check if two random arrays A and B are equal (shape + values)

In [ ]:
A = np.random.randint(0, 2, 5)
B = np.random.randint(0, 2, 5)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.array_equal(A, B)
```
</details>


**Q32.** Make an array immutable (read-only)

In [ ]:
Z = np.zeros(10)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
Z.flags.writeable = False
```
</details>


**Q33.** Convert (10,2) cartesian coords to polar (R, theta)

In [ ]:
Z = np.random.random((10, 2))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
R = np.sqrt(Z[:,0]**2 + Z[:,1]**2)
T = np.arctan2(Z[:,1], Z[:,0])
```
</details>


**Q34.** Find the closest value to a scalar v in a vector Z

In [ ]:
Z = np.arange(100)
v = np.random.uniform(0, 100)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
Z[(np.abs(Z - v)).argmin()]
```
</details>


**Q35.** Extract integer part of a positive float array (4 different methods)

In [ ]:
Z = np.random.uniform(0, 10, 10)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
Z - Z%1; Z//1; np.floor(Z); np.trunc(Z)
```
</details>


**Q36.** What is `np.ndenumerate`? Use it to print (index, value) for a 3x3 array

In [ ]:
Z = np.arange(9).reshape(3, 3)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
for index, value in np.ndenumerate(Z): print(index, value)
```
</details>


**Q37.** Create a 2D Gaussian-like array (10x10, centered, sigma=1)

In [ ]:
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
X, Y = np.meshgrid(np.linspace(-1,1,10), np.linspace(-1,1,10))
D = np.sqrt(X**2 + Y**2)
G = np.exp(-(D**2) / 2)
```
</details>


**Q38.** Randomly place p ones in an n×n array of zeros

In [ ]:
n, p = 10, 3
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
Z = np.zeros((n,n))
np.put(Z, np.random.choice(range(n*n), p, replace=False), 1)
```
</details>


**Q39.** Tell if a 2D array has null (all-zero) columns

In [ ]:
Z = np.random.randint(0, 3, (3, 10))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
(~Z.any(axis=0)).any()
```
</details>


**Q40.** Sum over the last two axes of a (3,4,3,4) array at once

In [ ]:
A = np.random.randint(0, 10, (3, 4, 3, 4))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
A.sum(axis=(-2, -1))
```
</details>


### Group H: Advanced (Q41–Q60) — continue in [NumPy 100](https://github.com/rougier/numpy-100)

**Q41.** Get the diagonal of A @ B without computing the full product

In [ ]:
A = np.random.uniform(0, 1, (5, 5))
B = np.random.uniform(0, 1, (5, 5))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.einsum('ij,ji->i', A, B)
```
</details>


**Q42.** Interleave 3 zeros between each value of [1,2,3,4,5]

In [ ]:
Z = np.array([1, 2, 3, 4, 5])
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
Z0 = np.zeros(len(Z) + (len(Z)-1)*3)
Z0[::4] = Z
```
</details>


**Q43.** Multiply (5,5,3) by (5,5) using broadcasting

In [ ]:
A = np.ones((5, 5, 3))
B = 2 * np.ones((5, 5))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
A * B[:, :, np.newaxis]
```
</details>


**Q44.** Swap two rows of an array

In [ ]:
A = np.arange(25).reshape(5, 5)
# Swap rows 0 and 1
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
A[[0, 1]] = A[[1, 0]]
```
</details>


**Q45.** Compute sliding-window mean of size 3 over a 1D array

In [ ]:
Z = np.arange(20)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
from numpy.lib.stride_tricks import sliding_window_view
sliding_window_view(Z, 3).mean(axis=-1)
```
</details>


**Q46.** Compute matrix rank of a random 10x10 matrix

In [ ]:
Z = np.random.uniform(0, 1, (10, 10))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.linalg.matrix_rank(Z)
```
</details>


**Q47.** Find the most frequent value in an array

In [ ]:
Z = np.random.randint(0, 10, 50)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.bincount(Z).argmax()
```
</details>


**Q48.** Extract unique rows from a 2D array

In [ ]:
Z = np.random.randint(0, 2, (6, 3))
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.unique(Z, axis=0)
```
</details>


**Q49.** Get the n largest values of an array (fast method)

In [ ]:
Z = np.arange(10000)
np.random.shuffle(Z)
n = 5
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
Z[np.argpartition(-Z, n)[:n]]
```
</details>


**Q50.** Write einsum for: sum(A), A*B, inner(A,B), outer(A,B)

In [ ]:
A = np.random.uniform(0, 1, 10)
B = np.random.uniform(0, 1, 10)
# YOUR CODE HERE


<details>
<summary>💡 Solution</summary>

```python
np.einsum('i->', A); np.einsum('i,i->i', A, B); np.einsum('i,i', A, B); np.einsum('i,j->ij', A, B)
```
</details>


### Group I: Q51–Q65

**Q51.** Use `np.fromiter` to build an array from a generator

In [ ]:
def gen(): yield from range(10)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.fromiter(gen(), dtype=float, count=-1)
```</details>


**Q52.** Compute point-by-point distances for (10,2) coordinates

In [ ]:
Z = np.random.random((10, 2))
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
X,Y = np.atleast_2d(Z[:,0], Z[:,1]); np.sqrt((X-X.T)**2 + (Y-Y.T)**2)
```</details>


**Q53.** Negate a boolean array in-place

In [ ]:
Z = np.random.randint(0, 2, 100)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.logical_not(Z, out=Z)
```</details>


**Q54.** Given bincount C, produce A such that np.bincount(A)==C

In [ ]:
C = np.bincount([1,1,2,3,4,4,6])
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.repeat(np.arange(len(C)), C)
```</details>


**Q55.** Compute means of subsets of D using index vector S (bincount with weights)

In [ ]:
D = np.random.uniform(0, 1, 100)
S = np.random.randint(0, 10, 100)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.bincount(S, weights=D) / np.bincount(S)
```</details>


**Q56.** Add 1 to elements of Z at indices I (handle repeated indices)

In [ ]:
Z = np.ones(10)
I = np.random.randint(0, 10, 20)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
Z += np.bincount(I, minlength=len(Z))
```</details>


**Q57.** Print min/max for np.int8, np.float32

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.iinfo(np.int8).min, np.iinfo(np.int8).max; np.finfo(np.float32).min, np.finfo(np.float32).max
```</details>


**Q58.** Read CSV with missing values (use genfromtxt)

In [ ]:
from io import StringIO
s = StringIO('1,2,3\n4,,5\n,6,7')
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.genfromtxt(s, delimiter=',', filling_values=0)
```</details>


**Q59.** Create rolling windows of size 3 from [1,2,...,14]

In [ ]:
Z = np.arange(1, 15)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
from numpy.lib.stride_tricks import sliding_window_view; sliding_window_view(Z, 4)
```</details>


**Q60.** Extract all 3x3 blocks from a 10x10 matrix

In [ ]:
Z = np.random.randint(0, 5, (10, 10))
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
sliding_window_view(Z, (3, 3))
```</details>


### Group J: Q61–Q75

**Q61.** Build cartesian product of [1,2,3] x [4,5]

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.transpose([np.tile([1,2,3], 2), np.repeat([4,5], 3)])
```</details>


**Q62.** Extract rows with unequal values from (10,3) matrix

In [ ]:
Z = np.random.randint(0, 5, (10, 3))
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
Z[Z.max(axis=1) != Z.min(axis=1)]
```</details>


**Q63.** Convert ints to binary matrix (each row = binary repr)

In [ ]:
I = np.array([0, 1, 2, 3, 15, 16])
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
((I.reshape(-1,1) & (2**np.arange(8))) != 0).astype(int)[:,::-1]
```</details>


**Q64.** Bootstrap 95% CI for mean of 1D array

In [ ]:
X = np.random.randn(100)
N = 1000
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
idx = np.random.randint(0, X.size, (N, X.size)); np.percentile(X[idx].mean(axis=1), [2.5, 97.5])
```</details>


**Q65.** Create structured array with x,y coords on [0,1]x[0,1]

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
Z = np.zeros((5,5), [('x',float),('y',float)]); Z['x'], Z['y'] = np.meshgrid(np.linspace(0,1,5), np.linspace(0,1,5))
```</details>


### Group K: Q66–Q80

**Q66.** Construct Cauchy matrix C[i,j] = 1/(x[i]-y[j])

In [ ]:
X, Y = np.arange(8), np.arange(8) + 0.5
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
1.0 / np.subtract.outer(X, Y)
```</details>


**Q67.** Block-sum 16x16 array (4x4 blocks)

In [ ]:
Z = np.ones((16, 16)); k = 4
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
Z.reshape(4,4,4,4).sum((1,3))
```</details>


**Q68.** Sum of p matrix-vector products: M (p,n,n), V (p,n,1) → (n,1)

In [ ]:
p, n = 10, 20
M = np.ones((p, n, n)); V = np.ones((p, n, 1))
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.tensordot(M, V, axes=[[0,2],[0,1]])
```</details>


**Q69.** Count unique colors in (256,256,3) uint8 image

In [ ]:
I = np.random.randint(0, 4, (16, 16, 3), dtype=np.uint8)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
len(np.unique(I.reshape(-1, 3), axis=0))
```</details>


**Q70.** Accumulate X into F by index I: F[I[i]] += X[i]

In [ ]:
X = [1,2,3,4,5,6]; I = [1,3,9,3,4,1]
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.bincount(I, X)
```</details>


**Q71.** Create RGBA dtype (4 unsigned bytes)

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.dtype([('r',np.ubyte),('g',np.ubyte),('b',np.ubyte),('a',np.ubyte)])
```</details>


**Q72.** Get yesterday, today, tomorrow as datetime64

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.datetime64('today')-1, np.datetime64('today'), np.datetime64('today')+1
```</details>


**Q73.** Create checkerboard 8x8 using np.tile

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.tile(np.array([[0,1],[1,0]]), (4,4))
```</details>


**Q74.** Create 3x3x3 random array; find min and max

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
Z = np.random.random((3,3,3)); Z.min(), Z.max()
```</details>


**Q75.** Sum small array faster than np.sum (use np.add.reduce)

In [ ]:
Z = np.arange(10)
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.add.reduce(Z)
```</details>


**Q76.** What does 0*np.nan, np.nan==np.nan, np.inf>np.nan evaluate to?

In [ ]:
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
nan, False, False. NaN != NaN; inf > nan is False.
</details>


**Q77.** Print all values of a large array (no truncation)

In [ ]:
Z = np.zeros((40, 40))
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.set_printoptions(threshold=np.inf); print(Z)
```</details>


**Q78.** Compute ((A+B)*(-A/2)) in-place (no copy)

In [ ]:
A = np.ones(3); B = np.ones(3)*2
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.add(A,B,out=B); np.divide(A,2,out=A); np.negative(A,out=A); np.multiply(A,B,out=A)
```</details>


**Q79.** Find rows of A (8,3) that contain elements of each row of B (2,2)

In [ ]:
A = np.random.randint(0,5,(8,3)); B = np.random.randint(0,5,(2,2))
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
C = (A[...,np.newaxis,np.newaxis]==B); np.where(C.any((3,1)).all(1))[0]
```</details>


**Q80.** Create record array from regular array

In [ ]:
Z = np.array([('Hello',2.5,3), ('World',3.6,2)])
# YOUR CODE HERE


<details><summary>💡 Solution</summary>
```python
np.rec.fromarrays(Z.T, names='c1,c2,c3', formats='U5,f8,i8')
```</details>


_You now have **80+ exercises** (Q1–Q80) with solution dropdowns. Q81–Q100: [NumPy 100](https://github.com/rougier/numpy-100)._

---
## Session 1 Reflection

**What concept was most surprising?**

_Your answer..._

**What still feels fuzzy?**

_Your answer..._

**Connection:** In Session 3 we'll use the same "batch @ weights" idea for linear regression predictions. In Session 5 we assemble everything into one model.